# Cấu hình đường dẫn vào nơi chứa các module để tiến hành chạy full pipeline

In [ ]:
import sys
import os

# Khai báo đường dẫn module
module_path = "/content/drive/MyDrive/se365/flow_SE365/git/src"
# if module_path not in sys.path:
#     sys.path.append(module_path)

os.chdir(module_path)

In [ ]:
!ls

config	  extract     main.ipynb   requirements.txt  rewrite
data	  fusion      pipeline.py  rerank	     setup.py
database  generation  __pycache__  retrieval


# Cài đặt các thư viện và các module

In [ ]:
!pip install -r requirements.txt

In [ ]:
from pipeline import qa_pipeline

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[RERANK] FP16 enabled on GPU
[RERANK] FP16 enabled on GPU


Device set to use cuda:0


[EXTRACT] Model loaded


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Chạy

In [ ]:
result = qa_pipeline(
    user_query="Người lao động phải được trả lương thế nào trong thời gian thử việc?",

    # rewrite
    use_rewrite = True,
    num_rewrites = 3,

    # retrieval
    retrieval_top_k = 100,  # -> 400

    # rewrite rrf
    use_rewrite_rrf = True,  # -> 50
    rewrite_rrf_top_n = 50,

    # rerank
    use_rerank = True, # -> 10

    # dual rerank
    use_dual_rerank = True, # mỗi rerank -> 50
    rerank_top_k = 10, # -> fusion 2 rerank lại -> 10

    # extract
    use_extract = True,

    # print result (in thông số khi pipeline thực hiện từng bước)
    print_result = True
)

[REWRITE] 4 queries
[HYBRID SEARCH] 4 done
[REWRITE RRF] 50 docs
[RERANK] dual done 10
[EXTRACT] done
[LLM] done




QUERY: Người lao động phải được trả lương thế nào trong thời gian thử việc?
REWRITES: ['Người lao động phải được trả lương thế nào trong thời gian thử việc?', 'Người lao động có quyền nhận lương trong thời gian thử việc như thế nào?', 'Mức lương người lao động nhận trong thời gian thử việc theo quy định pháp luật là bao nhiêu?', 'Người lao động được trả lương thế nào trong khoảng thời gian thử việc?']
RESULT:
------------------------------
Rank: 1
ID: 45_2019_QH14_D26
Hybrid_RRF: 0.01639344262295082
Rewrite_RRF: 0.06557377049180328
Rerank: None
Rerank_ViNLI: 0.734375
Rerank_Combine: 0.7353515625
Rerank_RRF: 0.03278688524590164
Extract: 0.6057440042495728
Text: Tiền lương thử việc Tiền lương của người lao động trong thời gian thử việc do hai bên thỏa thuận nhưng ít nhất phải bằng 85% mức lương của công việc đó.
Text_tokenized: Tiền_lương thử việc Tiền_lương của người lao_

In [ ]:
print(result["answer"])

Người lao động trong thời gian thử việc được trả lương ít nhất bằng 85% mức lương công việc, do hai bên thỏa thuận.
